# **Feature Engineering**

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [3]:
# 1. Carga de datos
df = pd.read_csv("../data/processed/Adidas_US_Sales_procesed.csv")

In [4]:
# 2. Feature Engineering temporal
df['Invoice Date'] = pd.to_datetime(df['Invoice Date'])
df['Month'] = df['Invoice Date'].dt.month
df['Year'] = df['Invoice Date'].dt.year
df['DayOfWeek'] = df['Invoice Date'].dt.dayofweek
df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(int)

In [5]:
# 3. DICCIONARIOS DE CATEGORÍAS (Orden explícito)
retailer = ["Foot Locker", "West Gear", "Sports Direct", "Kohl's", "Amazon", "Walmart"]
region = ["West", "Northeast", "Midwest", "South", "Southeast"]
product_cats = ["Men's Street Footwear", "Men's Athletic Footwear", "Men's Apparel",
                "Women's Street Footwear", "Women's Apparel", "Women's Athletic Footwear"]
sales_method = ["Online", "Outlet", "In-store"]

In [6]:
# 4. Aplicar OneHotEncoder con las categorías explícitas
categorical_cols = ['Retailer', 'Region', 'Product', 'Sales Method']
ohe = OneHotEncoder(categories=[retailer, region, product_cats, sales_method], 
                    sparse_output=False, handle_unknown='ignore')

# Transformar y crear un DataFrame con las nuevas columnas dummy
ohe_encoded = ohe.fit_transform(df[categorical_cols])
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
df_ohe = pd.DataFrame(ohe_encoded, columns=ohe_feature_names, index=df.index)

In [7]:
# 5. Seleccionar columnas numéricas y unirlas con las dummy
numeric_cols = ['Price per Unit', 'Month', 'Year', 'DayOfWeek', 'Is_Weekend']
df_numeric = df[numeric_cols].copy()

# Unir todo en el DataFrame final de features (X)
X = pd.concat([df_numeric, df_ohe], axis=1)
y = df['Total Sales']

In [8]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Price per Unit                     9648 non-null   float64
 1   Month                              9648 non-null   int32  
 2   Year                               9648 non-null   int32  
 3   DayOfWeek                          9648 non-null   int32  
 4   Is_Weekend                         9648 non-null   int64  
 5   Retailer_Foot Locker               9648 non-null   float64
 6   Retailer_West Gear                 9648 non-null   float64
 7   Retailer_Sports Direct             9648 non-null   float64
 8   Retailer_Kohl's                    9648 non-null   float64
 9   Retailer_Amazon                    9648 non-null   float64
 10  Retailer_Walmart                   9648 non-null   float64
 11  Region_West                        9648 non-null   float64
 12  Reg

In [9]:
X.columns

Index(['Price per Unit', 'Month', 'Year', 'DayOfWeek', 'Is_Weekend',
       'Retailer_Foot Locker', 'Retailer_West Gear', 'Retailer_Sports Direct',
       'Retailer_Kohl's', 'Retailer_Amazon', 'Retailer_Walmart', 'Region_West',
       'Region_Northeast', 'Region_Midwest', 'Region_South',
       'Region_Southeast', 'Product_Men's Street Footwear',
       'Product_Men's Athletic Footwear', 'Product_Men's Apparel',
       'Product_Women's Street Footwear', 'Product_Women's Apparel',
       'Product_Women's Athletic Footwear', 'Sales Method_Online',
       'Sales Method_Outlet', 'Sales Method_In-store'],
      dtype='str')

In [7]:
y.info()

<class 'pandas.Series'>
RangeIndex: 9648 entries, 0 to 9647
Series name: Total Sales
Non-Null Count  Dtype  
--------------  -----  
9648 non-null   float64
dtypes: float64(1)
memory usage: 75.5 KB


In [10]:
# 6. Guardado para los siguientes scripts
X.to_csv('../data/processed/X_preprocessed.csv', index=False)
y.to_csv('../data/processed/y_target.csv', index=False)

print("✅ Preprocesamiento con OneHotEncoder completado.")
print(f"Dimensiones de X: {X.shape} (Se expandieron las categorías a columnas binarias)")

✅ Preprocesamiento con OneHotEncoder completado.
Dimensiones de X: (9648, 25) (Se expandieron las categorías a columnas binarias)
